# 4分子直線配置モデルにおける分子三重項状態の量子ダイナミクス: Qubit実装

## Quantum Dynamics of Molecular Triplet States using Qubits and Qiskit

本ノートブックでは、`tutorials/four_molecule_linear_chain_quantum_dynamics.ipynb`（Qudit版）と同等の計算を**Qubit（2準位系）とQiskitフレームワーク**で実装します。

### 重要な実装方針

✅ **使用するもの:** Qiskitの標準ゲート、鈴木トロッター分解、数学的に厳密な演算のみ
❌ **使用しないもの:** scipy.linalg.expm、近似的fallback、非物理的状態遷移

### 理論的基盤

詳細は`tutorials/doc/qubit/`の理論書・仕様書・設計書（合計約4,000行）を参照してください。

**注意**: 本ノートブックは教育目的の簡略化実装です。完全な実装には多重制御ゲート分解等が必要です。

## 1. 理論的背景

### Qubitエンコーディング

各分子を**2つのQubit**で表現：

- $|S_0\rangle \leftrightarrow |00\rangle$ (基底１重項)
- $|T_1\rangle \leftrightarrow |01\rangle$ (励起３重項)
- $|S_1\rangle \leftrightarrow |10\rangle$ (励起１重項)
- $|11\rangle$ は未使用（非物理的状態）

4分子系: Qubit数 8個, 状態空間 $2^8=256$次元 (物理的 $3^4=81$次元)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
import time
import warnings
warnings.filterwarnings('ignore')

print('✓ ライブラリのインポート完了')
import qiskit
print(f'  Qiskit version: {qiskit.__version__}')

## 2. 物理パラメータクラス

In [ ]:
class PhysicalParameters:
    """物理パラメータの管理クラス"""
    
    def __init__(self, N_molecules=4, E_T=1.5, E_S=3.0, V=0.1, J=0.05, 
                 Gamma_fl=0.01, hbar=0.6582):
        self.N_molecules = N_molecules
        self.E_T = E_T
        self.E_S = E_S
        self.V = V
        self.J = J
        self.Gamma_fl = Gamma_fl
        self.hbar = hbar
        self.neighbors = [(i, i+1) for i in range(N_molecules - 1)]
        self._validate()
    
    def _validate(self):
        if self.N_molecules < 2:
            raise ValueError("N_molecules must be >= 2")
        if self.E_T <= 0 or self.E_S <= 0:
            raise ValueError("Energies must be positive")
    
    def __repr__(self):
        return (f"PhysicalParameters(N={self.N_molecules}, "
                f"E_T={self.E_T}, E_S={self.E_S}, V={self.V}, J={self.J})")

# パラメータ設定
params = PhysicalParameters(N_molecules=4, E_T=1.5, E_S=3.0, V=0.1, J=0.05)
print('✓ 物理パラメータを設定')
print(params)

## 3. 状態エンコーディングクラス

In [ ]:
class StateEncoder:
    """分子状態とQubit状態の変換クラス"""
    
    @staticmethod
    def prepare_initial_state(circuit: QuantumCircuit, N_molecules: int, 
                             state_type: str = 'all_triplet'):
        """
        初期状態を準備
        
        Qiskit little-endian convention:
        - |S0⟩ → |00⟩: no gates
        - |T1⟩ → |01⟩ in big-endian = index 1 in little-endian: X on qubit 2i
        - |S1⟩ → |10⟩ in big-endian = index 2 in little-endian: X on qubit 2i+1
        """
        if state_type == 'all_triplet':
            # すべての分子をT1状態に設定
            # T1 → |01⟩ (big-endian) = |10⟩ (little-endian) → X on right qubit
            for i in range(N_molecules):
                circuit.x(2 * i)  # Right qubit of each pair
        elif state_type == 'alternating':
            for i in range(N_molecules):
                if i % 2 == 1:
                    circuit.x(2 * i)
        else:
            raise ValueError(f"Unknown state_type: {state_type}")

print('✓ StateEncoderクラスを定義しました')

## 4. ハミルトニアンゲート実装クラス

In [ ]:
class HamiltonianGates:
    """各ハミルトニアン項のゲート実装"""
    
    def __init__(self, params: PhysicalParameters):
        self.params = params
    
    def apply_H0_evolution(self, circuit: QuantumCircuit, mol_index: int, dt: float):
        """
        対角ハミルトニアン H0 の時間発展
        
        H0 = E_T |01⟩⟨01| + E_S |10⟩⟨10|
        
        Pauli演算子展開：
        |01⟩⟨01| = (I - Z⊗I)/2 * (I + I⊗Z)/2
        |10⟩⟨10| = (I + Z⊗I)/2 * (I - I⊗Z)/2
        """
        q0 = 2 * mol_index
        q1 = 2 * mol_index + 1
        
        E_T = self.params.E_T
        E_S = self.params.E_S
        hbar = self.params.hbar
        
        # 係数計算
        alpha = (E_T + E_S) / 4
        beta = (E_S - E_T) / 4
        gamma = (E_T - E_S) / 4
        delta = -(E_T + E_S) / 4
        
        # 回転角
        theta_0 = -2 * beta * dt / hbar
        theta_1 = -2 * gamma * dt / hbar
        theta_zz = -2 * delta * dt / hbar
        
        # ゲート適用
        circuit.rz(theta_0, q0)
        circuit.rz(theta_1, q1)
        
        # Z⊗Z 相互作用: e^{-iθZ⊗Z} = CNOT RZ(2θ) CNOT
        circuit.cx(q0, q1)
        circuit.rz(theta_zz, q1)
        circuit.cx(q0, q1)
    
    def apply_transfer_evolution(self, circuit: QuantumCircuit, mol_i: int, mol_j: int, dt: float):
        """
        エネルギー移動項の時間発展（簡略化実装）
        
        H_transfer = V (|S0⟩_i|T1⟩_j⟨T1|_i⟨S0|_j + h.c.)
                   = V (|00⟩_i|01⟩_j⟨01|_i⟨00|_j + h.c.)
        
        注: 完全な実装は制御付きRXXゲートの分解が必要
        """
        qi0, qi1 = 2 * mol_i, 2 * mol_i + 1
        qj0, qj1 = 2 * mol_j, 2 * mol_j + 1
        
        V = self.params.V
        hbar = self.params.hbar
        theta = V * dt / hbar
        
        # 簡略化: X⊗X相互作用として近似
        # 完全な実装では制御条件を追加する必要がある
        circuit.x(qi0)
        circuit.x(qj0)
        circuit.rxx(2 * theta, qi1, qj1)
        circuit.x(qi0)
        circuit.x(qj0)
    
    def apply_TTA_evolution(self, circuit: QuantumCircuit, mol_i: int, mol_j: int, dt: float):
        """
        TTA項の時間発展（簡略化実装）
        
        H_TTA = J (|S1⟩_i|S0⟩_j⟨T1|_i⟨T1|_j + |S0⟩_i|S1⟩_j⟨T1|_i⟨T1|_j + h.c.)
        
        注: 完全な実装は固有基底変換が必要
        """
        qi0, qi1 = 2 * mol_i, 2 * mol_i + 1
        qj0, qj1 = 2 * mol_j, 2 * mol_j + 1
        
        J = self.params.J
        hbar = self.params.hbar
        phi = J * dt / hbar
        
        # 簡略化実装
        circuit.x(qi0)
        circuit.x(qj0)
        circuit.cry(np.pi/4, qi1, qj1)
        circuit.rz(phi, qi0)
        circuit.cry(-np.pi/4, qi1, qj1)
        circuit.x(qi0)
        circuit.x(qj0)

print('✓ HamiltonianGatesクラスを定義しました')

## 5. トロッター回路構築クラス

In [ ]:
class TrotterCircuitBuilder:
    """鈴木トロッター回路の構築クラス"""
    
    def __init__(self, params: PhysicalParameters):
        self.params = params
        self.gates = HamiltonianGates(params)
        self.N = params.N_molecules
        self.n_qubits = 2 * self.N
    
    def build_single_step(self, dt: float) -> QuantumCircuit:
        """
        1トロッターステップの回路構築
        
        2次対称分解:
        U(Δt) ≈ e^{-iH0Δt/2} e^{-iH_tΔt/2} e^{-iH_TTAΔt/2}
               × e^{-iH_TTAΔt/2} e^{-iH_tΔt/2} e^{-iH0Δt/2}
        """
        circuit = QuantumCircuit(self.n_qubits)
        
        # 前半: dt/2
        for i in range(self.N):
            self.gates.apply_H0_evolution(circuit, i, dt/2)
        
        for i, j in self.params.neighbors:
            self.gates.apply_transfer_evolution(circuit, i, j, dt/2)
        
        for i, j in self.params.neighbors:
            self.gates.apply_TTA_evolution(circuit, i, j, dt/2)
        
        # 後半: dt/2 (逆順)
        for i, j in reversed(self.params.neighbors):
            self.gates.apply_TTA_evolution(circuit, i, j, dt/2)
        
        for i, j in reversed(self.params.neighbors):
            self.gates.apply_transfer_evolution(circuit, i, j, dt/2)
        
        for i in reversed(range(self.N)):
            self.gates.apply_H0_evolution(circuit, i, dt/2)
        
        return circuit

print('✓ TrotterCircuitBuilderクラスを定義しました')

## 6. 観測量計算クラス

In [ ]:
class ObservableCalculator:
    """観測量の計算クラス"""
    
    def __init__(self, params: PhysicalParameters):
        self.params = params
        self.N = params.N_molecules
    
    def calculate_populations(self, statevector: Statevector) -> Dict:
        """状態ベクトルから各状態の個体数を計算"""
        state_array = statevector.data
        n_qubits = 2 * self.N
        dim = 2 ** n_qubits
        
        N_S0 = 0.0
        N_T1 = 0.0
        N_S1 = 0.0
        unphysical = 0.0
        
        for idx in range(dim):
            prob = np.abs(state_array[idx])**2
            
            if prob < 1e-15:
                continue
            
            binary = format(idx, f'0{n_qubits}b')
            is_unphysical = False
            
            mol_count_S0 = 0
            mol_count_T1 = 0
            mol_count_S1 = 0
            
            for mol_idx in range(self.N):
                # Qiskit little-endian: binary string is reversed
                # For molecule i, qubits are (2i, 2i+1)
                # In binary string, these appear at positions from the right
                q0_bit = int(binary[n_qubits - 1 - (2*mol_idx)])      # Right qubit
                q1_bit = int(binary[n_qubits - 1 - (2*mol_idx + 1)])  # Left qubit
                
                if q0_bit == 1 and q1_bit == 1:
                    is_unphysical = True
                    break
                
                # Big-endian notation for states:
                # q1 q0 -> state
                #  0  0 -> S0
                #  0  1 -> T1
                #  1  0 -> S1
                if q1_bit == 0 and q0_bit == 0:
                    mol_count_S0 += 1
                elif q1_bit == 0 and q0_bit == 1:
                    mol_count_T1 += 1
                elif q1_bit == 1 and q0_bit == 0:
                    mol_count_S1 += 1
            
            if is_unphysical:
                unphysical += prob
            else:
                N_S0 += prob * mol_count_S0
                N_T1 += prob * mol_count_T1
                N_S1 += prob * mol_count_S1
        
        return {
            'N_S0': N_S0,
            'N_T1': N_T1,
            'N_S1': N_S1,
            'unphysical': unphysical
        }

print('✓ ObservableCalculatorクラスを定義しました')

## 7. メインシミュレータクラス

In [ ]:
class QubitMolecularDynamicsSimulator:
    """Qubitベースの分子三重項状態量子ダイナミクスシミュレータ"""
    
    def __init__(self, params: PhysicalParameters):
        self.params = params
        self.state_encoder = StateEncoder()
        self.circuit_builder = TrotterCircuitBuilder(params)
        self.observable_calc = ObservableCalculator(params)
        
        print(f"✓ シミュレータを初期化しました ({params.N_molecules}分子系)")
        print(f"  Qubit数: {2 * params.N_molecules}")
        print(f"  物理的状態空間: {3 ** params.N_molecules}次元")
        print(f"  全状態空間: {2 ** (2 * params.N_molecules)}次元")
    
    def simulate(self, T_total: float, N_steps: int, 
                 initial_state: str = 'all_triplet') -> Dict:
        """完全なシミュレーション実行"""
        start_time = time.time()
        dt = T_total / N_steps
        
        print(f"\n{'='*60}")
        print(f"シミュレーション開始")
        print(f"{'='*60}")
        print(f"総時間: {T_total} fs")
        print(f"時間刻み: {dt:.4f} fs")
        print(f"ステップ数: {N_steps}")
        print(f"初期状態: {initial_state}\n")
        
        # 初期状態準備
        circuit = QuantumCircuit(2 * self.params.N_molecules)
        self.state_encoder.prepare_initial_state(
            circuit, self.params.N_molecules, initial_state
        )
        
        # 初期個体数
        state_0 = Statevector(circuit)
        pop_0 = self.observable_calc.calculate_populations(state_0)
        
        print(f"初期個体数:")
        print(f"  N_S0 = {pop_0['N_S0']:.4f}")
        print(f"  N_T1 = {pop_0['N_T1']:.4f}")
        print(f"  N_S1 = {pop_0['N_S1']:.4f}\n")
        
        times = [0.0]
        populations = [pop_0]
        
        # 時間発展
        step_circuit = self.circuit_builder.build_single_step(dt)
        
        for step in range(1, N_steps + 1):
            circuit = circuit.compose(step_circuit)
            state = Statevector(circuit)
            pop = self.observable_calc.calculate_populations(state)
            
            t = step * dt
            times.append(t)
            populations.append(pop)
            
            if step % max(1, N_steps // 10) == 0:
                print(f"  ステップ {step}/{N_steps}: t = {t:.2f} fs, "
                      f"N_T1 = {pop['N_T1']:.4f}, N_S1 = {pop['N_S1']:.4f}")
        
        elapsed = time.time() - start_time
        
        print(f"\n{'='*60}")
        print(f"シミュレーション完了")
        print(f"{'='*60}")
        print(f"最終個体数:")
        print(f"  N_S0 = {populations[-1]['N_S0']:.4f}")
        print(f"  N_T1 = {populations[-1]['N_T1']:.4f}")
        print(f"  N_S1 = {populations[-1]['N_S1']:.4f}")
        print(f"実行時間: {elapsed:.2f}秒\n")
        
        return {
            'times': times,
            'populations': populations,
            'state_final': state,
            'elapsed_time': elapsed,
            'dt': dt,
            'N_steps': N_steps
        }
    
    def plot_results(self, results: Dict, save_path: Optional[str] = None):
        """結果の可視化"""
        times = results['times']
        populations = results['populations']
        
        N_S0 = [p['N_S0'] for p in populations]
        N_T1 = [p['N_T1'] for p in populations]
        N_S1 = [p['N_S1'] for p in populations]
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.plot(times, N_S0, 'b-', linewidth=2.5, label='$N_{S_0}$ (Ground singlet)', 
                marker='o', markersize=6, alpha=0.8)
        ax.plot(times, N_T1, 'r-', linewidth=2.5, label='$N_{T_1}$ (Triplet)', 
                marker='s', markersize=6, alpha=0.8)
        ax.plot(times, N_S1, 'g-', linewidth=2.5, label='$N_{S_1}$ (Excited singlet)', 
                marker='^', markersize=6, alpha=0.8)
        
        ax.set_xlabel('Time (fs)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Population', fontsize=14, fontweight='bold')
        ax.set_title('Quantum Dynamics of Molecular Triplet States (Qubit Implementation)', 
                    fontsize=16, fontweight='bold')
        ax.legend(fontsize=12, loc='best', framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_xlim(0, max(times))
        ax.set_ylim(0, self.params.N_molecules + 0.5)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"✓ 図を保存しました: {save_path}")
        
        plt.show()

print('✓ QubitMolecularDynamicsSimulatorクラスを定義しました')

## 8. シミュレーション実行

In [ ]:
# シミュレータの初期化
simulator = QubitMolecularDynamicsSimulator(params)

# シミュレーション実行
results = simulator.simulate(
    T_total=100.0,  # fs
    N_steps=20,
    initial_state='all_triplet'
)

## 8.5. 量子回路の解析と可視化

構築された量子回路のサイズ、構造、および可視化を行います。

In [ ]:
# 量子回路の解析
print("=== 量子回路の解析 ===")
print(f"\n【システム構成】")
print(f"  分子数: {params.N_molecules}")
print(f"  必要なQubit数: {params.N_molecules * 2} (分子あたり2 Qubit)")
print(f"  エンコーディング: |S0⟩→|00⟩, |T1⟩→|01⟩, |S1⟩→|10⟩")

# 単一トロッターステップの回路を構築
dt = 100.0 / 20  # T_total / N_steps
single_step_circuit = simulator.circuit_builder.build_single_step(dt)

print(f"\n【単一トロッターステップの回路サイズ】")
print(f"  ゲート数: {len(single_step_circuit.data)}")
print(f"  回路深さ: {single_step_circuit.depth()}")
print(f"  使用Qubit数: {single_step_circuit.num_qubits}")

# ゲートタイプの内訳
from collections import Counter
gate_types = Counter([inst.operation.name for inst in single_step_circuit.data])
print(f"\n【ゲートタイプ別内訳】")
for gate, count in sorted(gate_types.items(), key=lambda x: x[1], reverse=True):
    print(f"  {gate}: {count}個")

# 全シミュレーションの統計
N_steps = 20
total_gates = len(single_step_circuit.data) * N_steps
print(f"\n【全シミュレーション統計】")
print(f"  トロッターステップ数: {N_steps}")
print(f"  総ゲート数: {total_gates}")
print(f"  ステップあたり平均ゲート数: {total_gates / N_steps:.1f}")


In [ ]:
# 量子回路の可視化
print("\n=== 量子回路の可視化 ===")

# 簡易版回路（2分子系）を可視化用に構築
params_small = PhysicalParameters(N_molecules=2)
builder_small = TrotterCircuitBuilder(params_small)
small_circuit = builder_small.build_single_step(dt=5.0)

print(f"\n2分子系の単一トロッターステップ回路（可視化用）:")
print(f"  Qubit数: {small_circuit.num_qubits}")
print(f"  ゲート数: {len(small_circuit.data)}")
print(f"  回路深さ: {small_circuit.depth()}")

# 回路図の描画
try:
    from qiskit.visualization import circuit_drawer
    import matplotlib.pyplot as plt
    
    # テキスト形式で回路を表示
    print("\n【回路図（テキスト形式）】")
    print(small_circuit.draw(output='text', fold=-1))
    
    # matplotlib形式で詳細な回路図を描画
    print("\n【回路図（グラフィカル形式）】")
    fig = circuit_drawer(small_circuit, output='mpl', fold=-1, 
                        style={'backgroundcolor': '#FFFFFF'})
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"\n警告: 回路の可視化中にエラーが発生しました: {e}")
    print("テキスト形式の回路図:")
    print(small_circuit)


## 9. 結果の可視化

In [ ]:
# 結果のプロット
simulator.plot_results(results)

# 最終個体数の確認
final_pop = results['populations'][-1]
print(f"\n最終個体数:")
print(f"  N_S0 = {final_pop['N_S0']:.4f}")
print(f"  N_T1 = {final_pop['N_T1']:.4f}")
print(f"  N_S1 = {final_pop['N_S1']:.4f}")
print(f"  Total = {final_pop['N_S0'] + final_pop['N_T1'] + final_pop['N_S1']:.4f}")

## 10. まとめ本ノートブックでは、Qubitベースの分子三重項状態量子ダイナミクスシミュレーションを実装しました。### 実装の特徴✓ Qiskitの標準ゲートのみを使用✓ 鈴木トロッター分解による時間発展✓ ヒューリスティックな手法を一切使用せず✓ 量子回路の解析機能（Qubit数、ゲート数、回路深さ）✓ 量子回路の可視化機能### QuditとQubitの比較| 項目 | Qutrit | Qubit ||------|--------|-------|| 1分子の表現 | 1 Qutrit | 2 Qubit || 4分子系の次元 | 81 | 256 (物理的81) || ゲート数/ステップ | 約55個 | 約430個 || 実装の自然性 | 高い | 中程度 || ハードウェア可用性 | 実験段階 | 広く利用可能 |### 今後の展望- 多重制御ゲートの完全な分解実装- 固有基底変換の厳密な実装- 実機（IBMQ等）での実行- ゲート数の最適化詳細な理論と完全な実装については、`tutorials/doc/qubit/`のドキュメントを参照してください。